# Agent liniowy — przepływ krok po kroku

Ten notebook najpierw pokazuje **jak działa** agent liniowy — uruchamia po kolei
umiejętności z `src/skills/*` i pokazuje wynik **każdego etapu** (to samo, co robi
`agent_linear/main.py`, ale interaktywnie)

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["LLM_MODEL"] = "gpt-5.4-mini" 
print("model:", os.environ.get("LLM_MODEL"), "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

from src.loader import load_all
from src.sources import prepare_input_texts
from src.skills.file_description.main import generate_file_description
from src.skills.tasks.main import generate_tasks
from src.skills.make_task.main import make_task
from src.skills.strategy.main import generate_strategy
from src.skills.document.main import generate_document

pipeline_usages = {}  # zużycie tokenów per etap (do policzenia kosztu)

GOAL = "Wygeneruj apelację z perspektywy obrony"
GOAL

model: gpt-5.4-mini | klucz z .env: True


'Wygeneruj apelację z perspektywy obrony'

## 0. Wczytanie akt

In [2]:
documents = load_all()
print(f"Wczytano {len(documents)} dokumentów")
for d in documents:
    print(" -", d.filename)

Wczytano 16 dokumentów
 - 02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf
 - 03_Protokół_z_badania_stanu_trzeźwości_analizatorem_wydechu.pdf
 - 04_Postanowienie_o_wszczęciu_dochodzenia.pdf
 - 05_Protokół_przesłuchania_świadka.pdf
 - 06_Protokół_przesłuchania_świadka.pdf
 - 07_Protokół_przesłuchania_świadka.pdf
 - 08_Postanowienie_o_przedstawieniu_zarzutów.pdf
 - 09_Protokół_przesłuchania_podejrzanego.pdf
 - 10_Protokół_przesłuchania_podejrzanego.pdf
 - 11_Akt_oskarżenia.pdf
 - 12_Protokół_rozprawy_głównej.pdf
 - 13_Odpowiedź_na_wezwanie_Sądu_Rejonowego_dotyczące_zarządzania_cmentarzem.pdf
 - 14_Opinia_biegłego_sądowego_z_zakresu_toksykologii_sądowej.pdf
 - 15_Protokół_rozprawy_głównej.pdf
 - 16_Wyrok.pdf
 - 17_Wniosek_o_sporządzenie_uzasadnienia.pdf


In [3]:
documents[0]

Document(filename='02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf', text='Komisariat Policji w Balicach Balice, 31 grudnia 2024 r.\nsierż. Wiesław Wilk\nNOTATKA URZĘDOWA\nW dniu datowania pełniłem służbę w obchodzie z post. Sylwią Sikorą na terenie\nKP Balice.\nOk. godz. 10.40 z polecenia dyżurnego KP Balice udaliśmy się do miejscowości\nSzczyglice w rejon cmentarza, gdzie według zgłoszenia kierujący pojazdem marki Jeep\nWrangler może być nietrzeźwy. Na miejscu zastano zgłaszającego: Karola Kota (Kot), który\noświadczył, że widział, jak kierujący pojazdem na terenie należącym do cmentarza usiłował\nzawrócić pojazd w alejce. Zgłaszający stwierdził, że kierowca prawdopodobnie jest\nnietrzeźwy. Następnie Karol Kot wskazał nam pojazd i udał się na pogrzeb. W pojeździe\nwskazanym przez Karola Kota na miejscu kierowcy zastano mężczyznę, od którego\nwyczuwalna była silna woń alkoholu. Wyciągnięto kluczyki ze stacyjki. Mężczyzną okazał\nsię być:\nDaniel Dzik, s. Jana i 

## 1. `generate_file_description` — opis każdego dokumentu

Surowe akta → zwięzłe, ukierunkowane na cel opisy ("mapa" sprawy).

In [4]:
from src.llm import track_usage

with track_usage() as _u:
    described = [generate_file_description(doc, GOAL) for doc in documents]
pipeline_usages["file_description"] = _u

In [5]:
described[0]

DescribedFile(title='NOTATKA URZĘDOWA', description='Dokument zawiera szczegółowy opis interwencji Policji wobec Daniela Dzika, którego zatrzymano jako kierującego pojazdem Jeep Wrangler nr rej. KRA 56789 po zgłoszeniu o możliwej nietrzeźwości. W notatce wskazano wyniki badania na zawartość alkoholu w wydychanym powietrzu: 1,63 mg/l, 1,57 mg/l, następnie 1,34 mg/l i 1,22 mg/l, a także informację o elektronicznym zatrzymaniu prawa jazdy nr 00123/00/1234567. Z perspektywy obrony dokument jest istotny, ponieważ zawiera dane o przebiegu czynności, czasie badań, przekazaniu pojazdu żonie oraz oświadczeniu Daniela Dzika co do spożycia alkoholu.', name='02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf')

## 2. `generate_tasks` — plan analizy

Każdy krok wskazuje, **które** dokumenty są mu potrzebne (`input`).

In [6]:
with track_usage() as _u:
    tasks = generate_tasks(GOAL, described)
pipeline_usages["generate_tasks"] = _u

for i, t in enumerate(tasks.thoughts, 1):
    print(f"[{i}] {t.action}")
    print(f"    dokumenty: {t.input}")
    print(f"    rozumowanie: {t.reasoning}")
    print()

[1] Przeanalizować zarzut prowadzenia pojazdu w stanie nietrzeźwości: kto prowadził, kiedy i gdzie doszło do interwencji, jakie były wyniki badań, czy zachowano prawidłowy tok czynności oraz jakie elementy mogą podważać ustalenia co do sprawstwa i stanu nietrzeźwości.
    dokumenty: ['02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf', '03_Protokół_z_badania_stanu_trzeźwości_analizatorem_wydechu.pdf', '04_Postanowienie_o_wszczęciu_dochodzenia.pdf', '08_Postanowienie_o_przedstawieniu_zarzutów.pdf', '09_Protokół_przesłuchania_podejrzanego.pdf', '10_Protokół_przesłuchania_podejrzanego.pdf']
    rozumowanie: Najpierw trzeba ustalić pełny obraz czynu z art. 178a § 1 k.k., bo od tego zależy ocena podstaw skazania, wiarygodności dowodów i ewentualnych uchybień w ustaleniach faktycznych. Kluczowe będą dokumenty z interwencji, badania trzeźwości, wszczęcia dochodzenia, zarzutów i wyjaśnień podejrzanego.

[2] Przeanalizować zarzut zniszczenia ławki: ustalić, co dokładnie zos

## 3. `make_task` — wykonanie kroków na wybranych dokumentach

Sedno: `prepare_input_texts` daje krokowi **tylko** wskazane dokumenty (selektywny kontekst).

In [7]:
from src.llm import track_usage

task_outputs = []
with track_usage() as _u:
    for i, task in enumerate(tasks.thoughts, 1):
        sources_text = prepare_input_texts(documents, task.input)
        out = make_task(GOAL, task, sources_text)
        task_outputs.append(out)
pipeline_usages["make_task"] = _u

In [9]:
task_outputs[0]

TaskOutput(output='Z dokumentów wynika, że zarzut z art. 178a § 1 k.k. oparto na interwencji z 31 grudnia 2024 r. w miejscowości Szczyglice, w rejonie cmentarza, na terenie alejki przy kaplicy/grobowcu rodziny Rysiów. Według notatki urzędowej policjanci ok. godz. 10.40 przybyli na miejsce po zgłoszeniu, że kierujący Jeepem Wranglerem może być nietrzeźwy. Zgłaszający Karol Kot oświadczył, że widział, jak kierujący usiłował zawrócić pojazd w alejce na terenie cmentarza, po czym wskazał samochód i odszedł. W pojeździe na miejscu kierowcy zastano Daniela Dzika, od którego wyczuwalna była silna woń alkoholu; wyjęto kluczyki ze stacyjki. Policja ustaliła tożsamość kierującego, sprawdziła go i pojazd jako nieposzukiwane, a następnie przeprowadziła badania trzeźwości.\n\nWyniki badań były wysokie i spójne z nietrzeźwością: o godz. 11.11 – 1,63 mg/l, o 11.28 – 1,57 mg/l, następnie po przewiezieniu na KP Balice o 12.21 – 1,34 mg/l i o 12.52 – 1,22 mg/l. Protokół wskazuje, że badanie wykonano urz

## 4. `generate_strategy` — wybór strategii

In [10]:
from src.llm import track_usage

with track_usage() as _u:
    strategy = generate_strategy(GOAL, task_outputs)
pipeline_usages["generate_strategy"] = _u

In [11]:
strategy

Strategy(reasoning='Najważniejszy zarzut w apelacji powinien dotyczyć czynu z art. 178a § 1 k.k., bo od jego prawidłowego ustalenia zależy zarówno sama odpowiedzialność karna, jak i większość najbardziej dolegliwych środków karnych (zakaz prowadzenia pojazdów, świadczenie pieniężne, przepadek pojazdu). Materiał dowodowy nie daje pełnej, bezpośredniej i niewątpliwej podstawy do przyjęcia, że oskarżony prowadził pojazd w stanie nietrzeźwości w chwili objętej zarzutem: świadek Karol Kot widział jedynie fragment manewrów, nie widział wjazdu ani całego przebiegu jazdy, Sylwia Sikora nie widziała prowadzenia ani spożycia alkoholu, a część jej relacji pochodziła z przekazu pośredniego. Dodatkowo brak jest obiektywnych dowodów momentu prowadzenia (monitoringu, śladów, pełnej dokumentacji ruchu pojazdu), a opinia biegłego nie pozwala retrospektywnie ustalić stężenia alkoholu o godzinie zdarzenia i ma charakter probabilistyczny. To czyni zarzut błędu w ustaleniach faktycznych oraz naruszenia art

## 5. `generate_document` — gotowa apelacja

In [12]:
from src.llm import track_usage

with track_usage() as _u:
    document = generate_document(GOAL, strategy, task_outputs)
pipeline_usages["generate_document"] = _u

print(document.tekst)

Sąd Okręgowy w Krakowie
za pośrednictwem
Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
II Wydział Karny

Sygn. akt: II K 25/25

Obrońca oskarżonego: r. pr. Lidia Lis
w imieniu oskarżonego Daniela Dzika

APELACJA
od wyroku Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie
z dnia 14 marca 2025 r.

Działając w imieniu oskarżonego Daniela Dzika, na podstawie art. 425 § 1 i 2 k.p.k., art. 427 § 1 i 2 k.p.k. oraz art. 444 k.p.k., zaskarżam powyższy wyrok w całości.

Zaskarżonemu wyrokowi zarzucam:

I. błąd w ustaleniach faktycznych przyjętych za podstawę orzeczenia, mający wpływ na jego treść, polegający na dowolnym przyjęciu, że oskarżony w chwili objętej zarzutem prowadził pojazd mechaniczny w ruchu lądowym oraz że znajdował się w stanie nietrzeźwości, podczas gdy materiał dowodowy nie daje pewnej podstawy do takiego ustalenia, w szczególności nie wykazuje w sposób jednoznaczny momentu prowadzenia pojazdu ani chwili spożycia alkoholu;

II. naruszenie art. 7, art. 410 i art. 424 k.p.k. pr

## 6. Koszt wygenerowania apelacji (cały pipeline)

Sumujemy zużycie tokenów ze wszystkich etapów agenta — koszt per etap i łącznie
(porównaj z baseline, gdzie to jeden wielki prompt).

In [13]:
from src.cost import estimate_cost

model = os.environ["LLM_MODEL"]
total_cost, total_calls, total_tok = 0.0, 0, 0
print(f"Koszt pipeline'u (model: {model}) — per etap:\n")
for name, u in pipeline_usages.items():
    c = estimate_cost(u, model)
    total_cost += c
    total_calls += u.calls
    total_tok += u.total_tokens
    print(f"  {name:18} {u.calls:2} wyw. | {u.total_tokens:7,} tok | ${c:.4f}")
print(f"\nRAZEM: {total_calls} wywołań | {total_tok:,} tok | ${total_cost:.4f}")

Koszt pipeline'u (model: gpt-5.4-mini) — per etap:

  file_description   16 wyw. |  28,465 tok | $0.0345
  generate_tasks      1 wyw. |   6,568 tok | $0.0127
  make_task           8 wyw. |  58,238 tok | $0.0793
  generate_strategy   1 wyw. |  12,332 tok | $0.0162
  generate_document   1 wyw. |  14,320 tok | $0.0238

RAZEM: 27 wywołań | 119,923 tok | $0.1664


In [14]:
from src.output import save_appeal

appeal = document.tekst
path = save_appeal(appeal, "agent_linear")
print("Zapisano apelację do:", path)

Zapisano apelację do: /Users/ewasuknarowska/Projects/WomenInTech/agent_linear/output/apelacja_2026-06-07_095458.txt
